**Import library untuk inferensi**

In [1]:
import json
import numpy as np
import requests
import base64
import tensorflow as tf

**Deklarasi endpoint dan data inferensi**

In [2]:
ENDPOINT_URL = "https://gym-exp-level.up.railway.app/v1/models/gym-experience-model:predict"

raw_data = {
    "Age": 25,
    "Gender": "Male",
    "Weight (kg)": 53.2,
    "Height (m)": 1.7,
    "Max_BPM": 190,
    "Avg_BPM": 164,
    "Resting_BPM": 56,
    "Session_Duration (hours)": 0.59,
    "Calories_Burned": 532.0,
    "Workout_Type": "Strength",
    "Fat_Percentage": 29.2,
    "Water_Intake (liters)": 2.8,
    "Workout_Frequency (days/week)": 3,
    "BMI": 14.39,
}

**Konversi data**

In [3]:
def make_example(data):
    feature = {}
    for key, value in data.items():
        if isinstance(value, str):
            feature[key] = tf.train.Feature(
                bytes_list=tf.train.BytesList(value=[value.encode("utf-8")])
            )
        elif isinstance(value, float):
            feature[key] = tf.train.Feature(
                float_list=tf.train.FloatList(value=[value])
            )
        elif isinstance(value, int):
            feature[key] = tf.train.Feature(
                int64_list=tf.train.Int64List(value=[value])
            )
    return tf.train.Example(features=tf.train.Features(feature=feature))

example_proto = make_example(raw_data)
serialize_proto_to_string = example_proto.SerializeToString()
base64_b64string = base64.b64encode(serialize_proto_to_string).decode("utf-8")

sample_data = {"instances": [{"examples": {"b64": base64_b64string}}]}

**Inferensi data ke API**

In [4]:
response = requests.post(ENDPOINT_URL, data=json.dumps(sample_data))

if response.status_code == 200:
    result = response.json()
    predictions = result["predictions"][0]

    class_mapping = {0: " ", 1: "Beginner", 2: "Intermediate", 3: "Expert"}

    predicted_index = np.argmax(predictions)
    predicted_label = class_mapping[predicted_index]
    confidence_score = predictions[predicted_index] * 100

    print(f"Hasil Prediksi Kelas: {predicted_label}")
    print(f"Confidence Score    : {confidence_score:.2f}%")

else:
    print(f"Inferensi Gagal! Status Code: {response.status_code}")
    print(f"Pesan Error dari Server: {response.text}")

Hasil Prediksi Kelas: Beginner
Confidence Score    : 100.00%
